In [ ]:
try:
    import os
    import base64
    import io
    from datetime import datetime
    import json
    import re
    import time
    import warnings
    from pathlib import Path

    import numpy as np
    import pandas as pd
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import mne
    import yasa
    import ipywidgets as widgets
    from IPython.display import display, HTML
    from ipyfilechooser import FileChooser
    from scipy.stats import kurtosis as scipy_kurtosis, skew as scipy_skew
    from scipy.signal import find_peaks, savgol_filter
except ImportError as e:
    print("⚠️ Import error, a package is missing:", e)
else:
    print("✅ Packages imported successfully!")

warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')

# Shared participant handling (tools/participant_selection_lib.py: exclusion registry + participant
# selector), probed from the repo root (the canonical launch folder) or from tools/, so the import works
# whatever folder Voila is launched from.
import sys
for _p in [os.getcwd(), os.path.join(os.getcwd(), 'tools'),
           os.path.join(os.path.dirname(os.getcwd()), 'tools'), os.path.join(os.getcwd(), '..', 'tools')]:
    if os.path.isfile(os.path.join(_p, 'participant_selection_lib.py')):
        if _p not in sys.path:
            sys.path.insert(0, _p)
        break
import participant_selection_lib as PSL

In [ ]:
# Signal-quality computation and figures, shared with tool 0: tools/quality_lib.py (found through the
# tools/ folder put on the path in the first cell). Edit the library, never a copy.
from quality_lib import (
    METRIC_INTERPRETATIONS, OVERVIEW_KEY_METRICS, OVERVIEW_METRIC_LABELS, build_avg_metrics_html,
    check_hypno_length, compute_channel_metrics, compute_channel_spectrograms, custom_stage_style,
    display_limits, hypno_vector, load_custom_stages, parse_custom_field, plot_histogram_figure,
    plot_median_hypnospectrogram, plot_overlay_distribution, plot_overlay_psd, plot_overlay_timeseries,
    plot_participant_electrode_scatter, plot_participant_pooled_boxplots, plot_psd_figure,
    plot_timeseries_figure, prepare_signal, render_hypnospectrogram,
)


def drop_suffix_duplicates(raw):
    """Keep only the -0 variant when MNE creates -0/-1 duplicates for repeated channel names."""
    groups = {}
    for ch in raw.ch_names:
        if ch.endswith('-0') or ch.endswith('-1'):
            base = ch.rsplit('-', 1)[0]
            groups.setdefault(base, []).append(ch)
    to_drop = []
    for base, ch_list in groups.items():
        if any(c.endswith('-0') for c in ch_list):
            to_drop.extend(c for c in ch_list if not c.endswith('-0'))
    if to_drop:
        raw.drop_channels(to_drop)
    return raw, to_drop


def adapt_remap_dict_to_suffixes(raw, remap_dict):
    """Handle MNE's -0 suffix when the remap config uses the base channel name."""
    ch_set = set(raw.ch_names)
    new_remap = {}
    for base_label, target in remap_dict.items():
        if base_label in ch_set:
            new_remap[base_label] = target
        elif f'{base_label}-0' in ch_set:
            new_remap[f'{base_label}-0'] = target
    return new_remap


def get_phys_bounds_uV(extras, ch_idx):
    """
    Reconstruct physical_min/max (in uV) from MNE _raw_extras.
    MNE 1.9 stores physical_max but not physical_min explicitly, and it keeps
    physical_max / offset in the channel's *native* EDF physical unit (uV, mV,
    V...), not in uV. extras['units'] is the multiplier from that native unit to
    volts (1e-6 for uV, 1e-3 for mV, 1.0 for V), so multiplying by units * 1e6
    brings the bounds into uV, matching raw.get_data() * 1e6. Without this, a
    channel declared in mV (e.g. Compumedics EOG/EMG/ECG, physical_max=1.0 mV)
    is compared against a 1.0 uV bound -- 1000x too small -- flagging ~100% of
    samples as 'at EDF bounds'.
    For symmetric 16-bit EDF: physical_min = -physical_max + 2*offset.
    Returns (phys_min_uV, phys_max_uV).
    """
    to_uV = float(extras['units'][ch_idx]) * 1e6
    phys_max = float(extras['physical_max'][ch_idx]) * to_uV
    offset = float(extras['offsets'][ch_idx]) * to_uV
    return -phys_max + 2 * offset, phys_max


def _fig_to_b64(fig):
    """Render a matplotlib figure to a base64-encoded PNG for HTML embedding."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=100, bbox_inches="tight")
    buf.seek(0)
    b64 = base64.b64encode(buf.read()).decode("utf-8")
    plt.close(fig)
    return b64


def generate_dataset_overview(df, reports_root, df_stage=None, custom_stages=None, excluded=None):
    """
    Generate dataset_overview.html from a quality_summary DataFrame.

    Global section:
      - stats table (all electrodes pooled)
      - mean/median by sleep stage table (if df_stage provided)
      - n_peaks frequency table by electrode
      - pooled distribution plot (one box per metric, with per-stage inset if df_stage provided)
      - grouped comparison plot (one box per metric per electrode)

    Per-electrode sections:
      - stats table for that electrode
      - distribution plot across participants (scatter colored red=flagged, grey=ok)

    Regenerated each run from the full cumulative quality_summary.tsv.
    `excluded` = {file_id: reason} of the participants left out of every statistic and figure (the
    caller already removed their rows; they are named in the header).
    """
    KEY_METRICS = OVERVIEW_KEY_METRICS
    ALL_NUMERIC = [
        "mean_uV", "std_uV", "kurtosis", "skewness",
        "p99_abs_uV", "p999_abs_uV",
        "flat_pct", "bounds_pct", "hist_extreme_pct",
    ]
    LABELS = OVERVIEW_METRIC_LABELS
    custom_stages = custom_stages or []
    _sy, SS_COLORS, _ytp, _ytl = custom_stage_style(custom_stages)
    STAGE_ORDER = ['W', 'N1', 'N2', 'N3', 'R'] + list(custom_stages)

    channels = sorted(df["channel"].unique())
    n_participants = int(df["file_id"].nunique())
    n_entries = len(df)
    n_flagged = int(df["exclude"].sum()) if "exclude" in df.columns else "?"
    n_flagged_part = int(df.groupby("file_id")["exclude"].any().sum()) if "exclude" in df.columns else "?"

    CSS = (
        "<style>"
        "body{font-family:Arial,sans-serif;max-width:1100px;margin:0 auto;padding:20px;color:#333}"
        "h1{color:#333;border-bottom:2px solid #ccc;padding-bottom:8px}"
        "h2{color:#333;margin-top:32px;border-bottom:1px solid #ddd;padding-bottom:4px}"
        "h3{color:#333;margin-top:16px}"
        "table{border-collapse:collapse;width:100%;font-size:.9em;margin-bottom:20px}"
        "th{background:#888;color:#fff;padding:6px 12px;text-align:left}"
        "td{padding:5px 12px;border-bottom:1px solid #eee}"
        "tr:nth-child(even){background:#f8f8f8}"
        ".flag{color:#c0392b;font-weight:bold}"
        "img{max-width:100%;margin:10px 0}"
        ".footer{color:#888;font-size:.8em;margin-top:40px;"
        "border-top:1px solid #ddd;padding-top:10px}"
        "</style>"
    )

    def _stats_table(data):
        HDR = (
            "<tr><th>Metric</th><th>Mean</th><th>Median</th>"
            "<th>p5</th><th>p25</th><th>p75</th><th>p95</th></tr>"
        )
        rows = []
        for m in ALL_NUMERIC:
            s = data[m].dropna() if m in data.columns else pd.Series(dtype=float)
            if s.empty:
                continue
            rows.append(
                f"<tr><td>{LABELS.get(m, m)}</td>"
                f"<td>{s.mean():.3g}</td><td>{s.median():.3g}</td>"
                f"<td>{s.quantile(0.05):.3g}</td><td>{s.quantile(0.25):.3g}</td>"
                f"<td>{s.quantile(0.75):.3g}</td><td>{s.quantile(0.95):.3g}</td></tr>"
            )
        return f"<table>{HDR}{''.join(rows)}</table>"

    def _stage_table():
        """Mean (median) by sleep stage — metrics as rows, stages as columns."""
        if df_stage is None or df_stage.empty:
            return ""
        STAGE_METRICS = ["mean_uV", "std_uV", "flat_pct", "bounds_pct", "hist_extreme_pct", "p99_abs_uV", "p999_abs_uV"]
        present_stages = [s for s in STAGE_ORDER if s in df_stage['stage'].values]
        if not present_stages:
            return ""
        hdr_stages = "".join(f"<th>{s}</th>" for s in present_stages)
        HDR = f"<tr><th>Metric</th>{hdr_stages}</tr>"
        rows = []
        for m in STAGE_METRICS:
            if m not in df_stage.columns:
                continue
            cells = []
            for stage in present_stages:
                vals = df_stage[df_stage['stage'] == stage][m].dropna()
                if vals.empty:
                    cells.append("<td>—</td>")
                else:
                    cells.append(f"<td>{vals.mean():.3g} ({vals.median():.3g})</td>")
            rows.append(f"<tr><td>{LABELS.get(m, m)}</td>{''.join(cells)}</tr>")
        if not rows:
            return ""
        return (
            "<h3>Mean (median) by sleep stage: all electrodes pooled</h3>"
            f"<table>{HDR}{''.join(rows)}</table>"
            "<p style='font-size:0.85em;color:#666;margin-top:-12px;'>"
            "Values pooled across all channels and participants for each stage. "
            "Format: mean (median).</p>"
        )

    def _npeaks_table(data):
        if "n_peaks" not in data.columns:
            return ""
        HDR = (
            "<tr><th>Electrode</th><th>1 peak (unimodal)</th>"
            "<th>2 peaks (bimodal)</th><th>≥3 peaks (multimodal)</th></tr>"
        )
        rows = []
        for ch in sorted(data["channel"].unique()):
            s = data[data["channel"] == ch]["n_peaks"]
            c1 = int((s == 1).sum())
            c2 = int((s == 2).sum())
            c3 = int((s >= 3).sum())
            f2 = ' class="flag"' if c2 else ""
            f3 = ' class="flag"' if c3 else ""
            rows.append(
                f"<tr><td>{ch}</td><td>{c1}</td>"
                f"<td{f2}>{c2}</td><td{f3}>{c3}</td></tr>"
            )
        return (
            "<h3>n_peaks distribution by electrode</h3>"
            f"<table>{HDR}{''.join(rows)}</table>"
        )

    def _scatter_with_flag(ax, x_center, vals, flags, jitter_width=0.07):
        """Scatter individual data points colored by flag status.
        Red = channel flagged for this participant; grey = within thresholds.
        showfliers=False must be set on the boxplot to avoid duplicate markers.
        """
        colors = ["#c0392b" if f else "#555555" for f in flags]
        jitter = np.random.default_rng(0).uniform(-jitter_width, jitter_width, len(vals))   # fixed seed: reproducible report
        for v, j, c in zip(vals, jitter, colors):
            ax.scatter(x_center + j, v, color=c, s=28, alpha=0.75, zorder=3,
                       linewidths=0)

    def _flag_legend(fig):
        """Add a flag/ok legend to the figure."""
        handles = [
            matplotlib.lines.Line2D(
                [0], [0], marker="o", color="w", markerfacecolor="#c0392b",
                markersize=8, label="Flagged",
            ),
            matplotlib.lines.Line2D(
                [0], [0], marker="o", color="w", markerfacecolor="#555555",
                markersize=8, label="Not flagged",
            ),
        ]
        fig.legend(handles=handles, loc="upper right", fontsize=9, framealpha=0.9)

    def _vals_and_flags(data, metric):
        """Return aligned (values, flags) arrays for rows where metric is not NaN."""
        valid = data[metric].notna()
        vals = data.loc[valid, metric].values
        if "exclude" in data.columns:
            flags = data.loc[valid, "exclude"].fillna(False).astype(bool).values
        else:
            flags = np.zeros(len(vals), dtype=bool)
        return vals, flags

    def _pooled_boxplots(data):
        """All electrodes combined — one box per metric.
        Each subplot contains a per-stage inset (upper-right corner) when stage data is available.
        Gives the dataset-wide distribution and the per-stage breakdown in a single compact view.
        """
        # Pre-compute stage info once for all subplots
        present_stages = []
        if df_stage is not None and not df_stage.empty:
            present_stages = [s for s in STAGE_ORDER if s in df_stage['stage'].values]

        ncols = min(3, len(KEY_METRICS))
        nrows = int(np.ceil(len(KEY_METRICS) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows), squeeze=False)
        axes = axes.flatten()
        for ax, m in zip(axes, KEY_METRICS):
            vals, flags = _vals_and_flags(data, m)
            if len(vals) == 0:
                ax.set_visible(False)
                continue
            bp = ax.boxplot(
                [vals], patch_artist=True, showfliers=False,
                medianprops=dict(color="#c0392b", linewidth=2),
                whiskerprops=dict(linewidth=1.2),
                capprops=dict(linewidth=1.2),
            )
            bp["boxes"][0].set_facecolor("#d4e8f5")
            bp["boxes"][0].set_alpha(0.8)
            _scatter_with_flag(ax, x_center=1, vals=vals, flags=flags)
            ax.set_title(LABELS.get(m, m), fontsize=11, fontweight="bold")
            ax.set_xticks([])
            for sp in ["right", "top"]:
                ax.spines[sp].set_visible(False)

            # Per-stage inset (upper-right). The main boxplot is pushed against the
            # y-axis by widening the x-limits to the right, so the box + points and the
            # inset never overlap while the left gap stays minimal. The box sits at x=1
            # (spans 0.75–1.25); lower bound 0.65 keeps it tight to the y-axis, upper
            # bound 2.6 frees the right ~half of the axes for the inset.
            if present_stages and df_stage is not None and m in df_stage.columns:
                ax.set_xlim(0.65, 2.6)
                ax_in = ax.inset_axes([0.42, 0.20, 0.56, 0.74])
                groups_in = [df_stage[df_stage['stage'] == s][m].dropna().values
                             for s in present_stages]
                bp_in = ax_in.boxplot(
                    groups_in, labels=present_stages, patch_artist=True, showfliers=False,
                    medianprops=dict(color='#333', linewidth=1.5),
                    whiskerprops=dict(linewidth=1.0),
                    capprops=dict(linewidth=1.0),
                )
                for patch, stage in zip(bp_in['boxes'], present_stages):
                    patch.set_facecolor(SS_COLORS[stage])
                    patch.set_alpha(0.85)
                # Keep the inset y-axis visible so per-stage absolute values stay readable.
                ax_in.tick_params(axis='x', labelsize=7)
                ax_in.tick_params(axis='y', labelsize=7)
                for sp in ['right', 'top']:
                    ax_in.spines[sp].set_visible(False)
                ax_in.patch.set_facecolor('#fafafa')
                ax_in.patch.set_alpha(0.9)

        for ax in axes[len(KEY_METRICS):]:
            ax.set_visible(False)
        _flag_legend(fig)
        fig.suptitle(
            f"All electrodes pooled: metric distributions (n={len(data)} channel records)",
            fontsize=12, y=1.02,
        )
        plt.tight_layout()
        return f'<img src="data:image/png;base64,{_fig_to_b64(fig)}" style="max-width:100%;">'

    def _grouped_boxplots(data):
        """One subplot per metric; x-axis = electrode — compares electrodes side by side.
        showfliers=False keeps scale anchored to the IQR range (no outlier points
        inflating the axis). Individual values are not shown here; see per-electrode
        sections for participant-level detail.
        """
        ncols = min(3, len(KEY_METRICS))
        nrows = int(np.ceil(len(KEY_METRICS) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows), squeeze=False)
        axes = axes.flatten()
        for ax, m in zip(axes, KEY_METRICS):
            groups = [data[data["channel"] == ch][m].dropna().values for ch in channels]
            bp = ax.boxplot(
                groups, labels=channels, patch_artist=True, showfliers=False,
                medianprops=dict(color="#c0392b", linewidth=2),
                whiskerprops=dict(linewidth=1.2),
                capprops=dict(linewidth=1.2),
            )
            for patch in bp["boxes"]:
                patch.set_facecolor("#d4e8f5")
                patch.set_alpha(0.8)
            ax.set_title(LABELS.get(m, m), fontsize=11, fontweight="bold")
            rot = 30 if len(channels) > 4 else 0
            ax.tick_params(axis="x", labelrotation=rot, labelsize=9)
            for sp in ["right", "top"]:
                ax.spines[sp].set_visible(False)
        for ax in axes[len(KEY_METRICS):]:
            ax.set_visible(False)
        plt.tight_layout()
        return f'<img src="data:image/png;base64,{_fig_to_b64(fig)}" style="max-width:100%;">'

    def _electrode_boxplots(ch_data, ch_name):
        """One subplot per metric; distribution of that metric across participants
        for this electrode. Each dot = one participant.
        Red dot = this channel was flagged for that participant.
        Grey dot = within all thresholds.
        showfliers=False removes matplotlib's default outlier markers so each
        participant appears exactly once as a colored dot.
        """
        ncols = min(3, len(KEY_METRICS))
        nrows = int(np.ceil(len(KEY_METRICS) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows), squeeze=False)
        axes = axes.flatten()
        for ax, m in zip(axes, KEY_METRICS):
            vals, flags = _vals_and_flags(ch_data, m)
            if len(vals) == 0:
                ax.set_visible(False)
                continue
            bp = ax.boxplot(
                [vals], patch_artist=True, showfliers=False,
                medianprops=dict(color="#c0392b", linewidth=2),
                whiskerprops=dict(linewidth=1.2),
                capprops=dict(linewidth=1.2),
            )
            bp["boxes"][0].set_facecolor("#d4e8f5")
            bp["boxes"][0].set_alpha(0.8)
            _scatter_with_flag(ax, x_center=1, vals=vals, flags=flags)
            ax.set_title(LABELS.get(m, m), fontsize=11, fontweight="bold")
            ax.set_xticks([])
            for sp in ["right", "top"]:
                ax.spines[sp].set_visible(False)
        for ax in axes[len(KEY_METRICS):]:
            ax.set_visible(False)
        _flag_legend(fig)
        fig.suptitle(
            f"Electrode {ch_name}: distribution across {len(ch_data)} participants",
            fontsize=12, y=1.02,
        )
        plt.tight_layout()
        return f'<img src="data:image/png;base64,{_fig_to_b64(fig)}" style="max-width:100%;">'

    # Build global section
    global_block = (
        "<h2>All electrodes: pooled statistics</h2>"
        f"<h3>Summary statistics (all channels combined, n={n_entries})</h3>"
        + _stats_table(df)
        + _stage_table()
        + _npeaks_table(df)
        + "<h3>Pooled distribution of each metric (all electrodes combined)</h3>"
        + _pooled_boxplots(df)
        + "<h3>Distributions by electrode (comparison)</h3>"
        + _grouped_boxplots(df)
    )

    # Build per-electrode sections
    electrode_blocks = ""
    for ch in channels:
        ch_data = df[df["channel"] == ch]
        electrode_blocks += (
            f"<h2>Electrode: {ch}</h2>"
            f"<h3>Statistics across {len(ch_data)} participants</h3>"
            + _stats_table(ch_data)
            + _electrode_boxplots(ch_data, ch)
        )

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
    html = (
        '<!DOCTYPE html><html lang="en"><head><meta charset="UTF-8">'
        "<title>Dataset Quality Overview</title>"
        f"{CSS}</head><body>"
        "<h1>Dataset Quality Overview</h1>"
        f"<p><b>Dataset:</b> {reports_root.parent.name}&nbsp;|&nbsp;"
        f"<b>Participants:</b> {n_participants}&nbsp;|&nbsp;"
        f"<b>Entries (channel × participant):</b> {n_entries}&nbsp;|&nbsp;"
        f"<b>Flagged channels:</b> {n_flagged}&nbsp;|&nbsp;"
        f"<b>Participants with ≥1 flag:</b> {n_flagged_part}</p>"
        + (f'<p style="color:#8b1a1a"><b>{len(excluded)} excluded participant(s) left out of every '
           f'statistic and figure</b> (still in quality_summary.tsv, participant_excluded = True): '
           + '; '.join(f'{k} ({v})' for k, v in excluded.items()) + '</p>' if excluded else '')
        + global_block
        + electrode_blocks
        + '<p class="footer">'
        f"Generated by 6_quality_overview_voila.ipynb: {timestamp} — "
        f"Source: quality_summary.tsv ({n_entries} entries, {n_participants} participants)"
        "</p></body></html>"
    )

    (reports_root / "dataset_overview.html").write_text(html, encoding="utf-8")


## Preprocessing, Phase 1: Quality Overview 

This tool computes **signal quality metrics per channel** for every EDF file in a folder. It produces:
- One **HTML report** per participant (histograms, PSDs, time series, descriptive stats, hypnograms/spectrograms)
- A **`quality_summary.tsv`** with all numeric metrics
- A **`dataset_overview.html`** with dataset-level statistics and distributions per electrode

**Instructions:**
1. Use the first file chooser to select your **study folder**.
2. Select the **remap/reref config JSON** produced by *select&amp;remap_channels_edf*.
3. Adjust the **hypnogram suffix** if needed and the **quality thresholds**.
4. Click **Run Analysis**.

Quality metrics, amplitude histograms, PSDs, and time series are all computed on the **raw, unfiltered signal**. Only the hypnospectrogram uses a 0.1&ndash;40&nbsp;Hz bandpass filter (applied only for that plot).

In [ ]:
fc_folder = FileChooser()
fc_folder.title = '<b>Select your study folder:</b>'
fc_folder.show_only_dirs = True

hypno_suffix = widgets.Text(
    value='_Hypnogram_remapped.txt',
    description='Hypno suffix:',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='420px')
)
hypno_suffix_info = widgets.HTML(value='')

custom_stages_box = widgets.Text(
    value='',
    description='Custom stages:',
    placeholder='non-AASM labels to keep, e.g. N4 (auto-filled from config)',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='420px')
)
custom_stages_info = widgets.HTML(value='')


existing_reports_info = widgets.HTML(value='')
acq_info = widgets.HTML(value='')  # sampling frequency + acquisition high-pass, from EDF headers

fc_config = FileChooser(filter_pattern='*.json')
fc_config.title = '<b>Select remap/reref config JSON</b> (from select&amp;remap_channels_edf):'


def read_edf_sf_highpass(path, keep_labels=None):
    """Read sampling frequency + acquisition high-pass straight from the EDF header bytes (same
    approach as 1_inspect_edf, no MNE). Restricted to the channels in `keep_labels` (the selected
    montage) when given, else all channels. Returns (sf_str, hp_str) from the most common value
    among the kept channels: sampling frequency in Hz, and high-pass from the 'prefiltering' field
    ('none/DC' when the HP tag is absent). Raises on an unreadable header."""
    with open(path, 'rb') as f:
        f.seek(244)                                   # offset of 'duration of a data record' (8 bytes)
        dur = float(f.read(8).decode('latin-1').strip())
        n = int(f.read(4).decode('latin-1').strip())  # number of signals (channels)
        labels = [f.read(16).decode('latin-1').strip() for _ in range(n)]  # channel labels, at offset 256
        f.seek(256 + n * 136)                         # skip label/transducer/dim/phys/dig -> prefiltering
        prefilt = [f.read(80).decode('latin-1').strip() for _ in range(n)]
        spr = [f.read(8).decode('latin-1').strip() for _ in range(n)]
    idx = list(range(n)) if keep_labels is None else [i for i in range(n) if labels[i] in keep_labels]
    # Sampling frequency = samples per data record / record duration (see SPEC / 1_inspect_edf).
    sfs = []
    for i in idx:
        try:
            v = float(spr[i]) / dur
            sfs.append(f'{int(v)}' if v == int(v) else f'{v:g}')
        except (ValueError, ZeroDivisionError):
            pass
    # High-pass from the 'prefiltering' field, e.g. 'HP:0.3Hz LP:70Hz' -> 0.3; absent -> none/DC.
    hps = []
    for i in idx:
        m = re.search(r'HP[:\s]*([\d.]+)', prefilt[i], re.IGNORECASE)
        hps.append(f'{float(m.group(1)):g} Hz' if m else 'none/DC')
    # Show ALL distinct values among the kept channels (joined by ' / '), NOT the most common one:
    # a montage whose EEG channels ended up at different rates (bad export) must be visible, not
    # hidden. The caller flags any ' / ' (i.e. mixed) entry with a warning.
    sf_str = (' / '.join(sorted(set(sfs), key=float)) + ' Hz') if sfs else '?'
    hp_str = ' / '.join(sorted(set(hps))) if hps else '?'
    return sf_str, hp_str


def update_existing_reports_info(chooser):
    if not chooser.selected:
        existing_reports_info.value = ''
        hypno_suffix_info.value = ''
        acq_info.value = ''
        fc_config.reset()
        return
    data_folder = Path(chooser.selected)
    _cs = load_custom_stages(data_folder)
    custom_stages_box.value = ', '.join(_cs)
    custom_stages_info.value = (
        f'<small style="color:#2e7d32;">Custom stage(s) loaded: {", ".join(_cs)}</small>' if _cs
        else '<small style="color:#888;">No custom stages registered for this dataset.</small>'
    )
    fc_config.reset(path=str(data_folder))
    edf_files = [f for f in sorted(data_folder.rglob('*')) if f.suffix.lower() == '.edf' and not f.name.startswith('._')]
    n_total = len(edf_files)
    if n_total == 0:
        existing_reports_info.value = '<small style="color:#888;">No EDF files found in selected folder (recursive scan).</small>'
        hypno_suffix_info.value = ''
        acq_info.value = ''
        return
    reports_root = data_folder / 'reports_quality_overview'
    # "Already done" = BOTH the HTML report AND the per-file metrics TSV present. A file with
    # only one of the two (run interrupted between them, or a manual delete) is counted as a
    # mismatch and will be reprocessed when Run is pressed.
    n_existing = 0
    n_mismatch = 0
    done_ids = []
    for f in edf_files:
        f_dir = reports_root / f.parent.relative_to(data_folder)
        has_report = (f_dir / f'{f.stem}_quality_overview.html').exists()
        has_data   = (f_dir / f'{f.stem}_quality_metrics.tsv').exists()
        if has_report and has_data:
            n_existing += 1
            done_ids.append(f.stem)
        elif has_report != has_data:
            n_mismatch += 1
    _reg, _reg_warn = PSL.load_registry(data_folder)
    participant_selector.set_participants(
        [f.stem for f in edf_files], done=done_ids,
        excluded={s['file_id']: s['reason'] for s in PSL.effective_status(_reg).values() if s['excluded']})
    _mismatch_txt = (f' <span style="color:#e67e22;">({n_mismatch} inconsistent, will be reprocessed)</span>'
                     if n_mismatch else '')
    existing_reports_info.value = (
        f'<small style="color:#555;">'
        f'{n_existing} / {n_total} participant(s) already fully processed (report + data).{_mismatch_txt}</small>'
    )
    if PSL.registry_newer_than(data_folder, reports_root / 'quality_summary.tsv'):
        existing_reports_info.value += PSL.stale_tables_html(PSL.STALE_RERUN)
    # --- Hypnogram suffix detection ---
    # For each EDF, count all .txt files whose name starts with the EDF stem (no break:
    # a participant can have both a raw and a remapped hypno).
    # Selection: among suffixes appearing for >=50% of the maximum count, prefer the
    # longest (more specific = remapped/processed version). The 50% threshold prevents
    # rare/accidental long suffixes from winning over the true candidate.
    # Tiebreaker when two candidates are equally long: highest count.
    all_txt = [f for f in data_folder.rglob('*') if f.suffix.lower() == '.txt']
    suffix_counts = {}
    for edf in edf_files:
        for txt in all_txt:
            if os.path.normcase(txt.name).startswith(os.path.normcase(edf.stem)):
                suffix = txt.name[len(edf.stem):]
                suffix_counts[suffix] = suffix_counts.get(suffix, 0) + 1
    if not suffix_counts:
        hypno_suffix_info.value = (
            '<small style="color:#e67e00;">No hypnogram files detected — '
            'check that hypnograms are present or adjust the suffix manually.</small>'
        )
    else:
        # Auto-selection skips event exports: Curry writes its scored events to
        # *_ScoredEvents_Export.txt, a .txt sitting next to the hypnograms whose suffix is
        # LONGER than _Hypnogram_remapped.txt, so 'prefer the longest' would select it and try
        # to read event lines as sleep stages. Excluding 'event' rather than requiring 'hypno'
        # leaves hypnogram naming unconstrained. Filtered out of the SELECTION only — every
        # .txt suffix stays listed below, so a wrong detection stays visible and correctable.
        sel_counts = {s: c for s, c in suffix_counts.items() if 'event' not in s.lower()}
        if not sel_counts:
            sel_counts = suffix_counts  # only event exports found: fall back rather than fail
        max_count = max(sel_counts.values())
        candidates = {s: c for s, c in sel_counts.items() if c >= max_count * 0.5}
        best_suffix, best_count = max(candidates.items(), key=lambda x: (len(x[0]), x[1]))
        hypno_suffix.value = best_suffix
        parts = [f'<b>{s}</b>&nbsp;(×{c}){"&nbsp;← selected" if s == best_suffix else ""}'
                 for s, c in sorted(suffix_counts.items(), key=lambda x: -x[1])]
        color = '#2e7d32' if best_count == n_total else '#e67e00'
        hypno_suffix_info.value = (
            f'<small style="color:{color};">Detected:&nbsp;{"&nbsp;·&nbsp;".join(parts)}'
            f'&nbsp;— {best_count}/{n_total} EDF files matched</small>'
        )

    # The acquisition line (sampling frequency / high-pass of the SELECTED channels) needs the
    # config JSON; it is filled by update_acq_info() once the config is chosen (fc_config was
    # just reset above), so here we only show a hint.
    acq_info.value = ('<small style="color:#888;">Select the config JSON above to list the '
                      'sampling frequency / high-pass of the selected channels.</small>')

fc_folder.register_callback(update_existing_reports_info)


def update_acq_info(*_):
    # Sampling frequency + acquisition high-pass of the SELECTED channels (the config 'remap' keys),
    # read straight from the EDF headers. Needs both the folder and the config JSON; grouped by
    # unique value with file counts. Registered on the config chooser below.
    if not (fc_folder.selected and fc_config.selected):
        acq_info.value = ('<small style="color:#888;">Select the study folder and the config JSON '
                          'to list the sampling frequency / high-pass of the selected channels.</small>')
        return
    try:
        data_folder = Path(fc_folder.selected)
        with open(fc_config.selected, 'r', encoding='utf-8') as _f:
            _cfg = json.load(_f)
        _edf_files = [f for f in sorted(data_folder.rglob('*')) if f.suffix.lower() == '.edf' and not f.name.startswith('._')]
        acq_combos = {}
        for _edf in _edf_files:
            _keep = set(_cfg.get(_edf.stem, {}).get('remap', {}).keys())
            try:
                _sf, _hp = read_edf_sf_highpass(_edf, _keep or None)
                _key = (_sf, _hp)
            except Exception:
                _key = ('unreadable', '')
            acq_combos[_key] = acq_combos.get(_key, 0) + 1
        # A montage with several distinct rates / high-passes (sf or hp contains ' / ') is shown
        # in red — usually a bad export (channels exported at different sampling frequencies).
        acq_parts, _any_mixed = [], False
        for (_sf, _hp), _c in sorted(acq_combos.items(), key=lambda kv: -kv[1]):
            if _sf == 'unreadable':
                acq_parts.append(f'{_c} unreadable')
                continue
            _mixed = (' / ' in _sf) or (' / ' in _hp)
            _any_mixed = _any_mixed or _mixed
            _txt = f'{_sf} · high-pass {_hp} (×{_c})'
            acq_parts.append(f'<b style="color:#c0392b;">⚠ {_txt}</b>' if _mixed else _txt)
        _note = (' &nbsp;&mdash;&nbsp; <b style="color:#c0392b;">⚠ a montage mixes several sampling '
                 'frequencies / high-passes (check the export!)</b>') if _any_mixed else ''
        acq_info.value = (
            '<small style="color:#555;">Selected channels (EDF header): '
            + '&nbsp;&nbsp;·&nbsp;&nbsp;'.join(acq_parts) + _note + '</small>'
        )
    except Exception as _acq_err:
        acq_info.value = f'<small style="color:#e67e00;">Could not read acquisition info: {_acq_err}</small>'

fc_config.register_callback(update_acq_info)

# Who runs: all participants or a subset, + 'Skip already processed' (shared selector, filled when the
# study folder is chosen). Excluded participants (registry) are processed too, only flagged.
participant_selector = PSL.ParticipantSelector()
skip_existing = participant_selector.cb_skip

thresh_flat = widgets.BoundedFloatText(
    value=3.5, min=0.0, max=100.0, step=0.5,
    description='flat_pct (%) >',
    style={'description_width': '160px'},
    layout=widgets.Layout(width='300px')
)
thresh_bounds = widgets.BoundedFloatText(
    value=1.0, min=0.0, max=100.0, step=0.1,
    description='bounds_pct (%) >',
    style={'description_width': '160px'},
    layout=widgets.Layout(width='300px')
)
thresh_peaks = widgets.BoundedIntText(
    value=2, min=2, max=10, step=1,
    description='n_peaks >=',
    style={'description_width': '160px'},
    layout=widgets.Layout(width='300px')
)
thresh_std_low = widgets.BoundedFloatText(
    value=5.0, min=0.0, max=500.0, step=0.5,
    description='std_uV (µV) <',
    style={'description_width': '160px'},
    layout=widgets.Layout(width='300px')
)
thresh_hist_extreme = widgets.BoundedFloatText(
    value=1.0, min=0.0, max=100.0, step=0.1,
    description='hist_extreme_pct (%) >',
    style={'description_width': '160px'},
    layout=widgets.Layout(width='300px')
)

# ---- Optional resampling (speeds up processing; off by default) ----
cb_resample = widgets.Checkbox(
    value=False, description='Activate resampling',
    style={'description_width': 'initial'}
)
txt_target_freq = widgets.BoundedIntText(
    value=256, min=1, max=10000, step=1,
    description='Target frequency (Hz) :',
    style={'description_width': '180px'},
    # Initial visibility follows the checkbox default (same reason as the high-pass box below).
    layout=widgets.Layout(width='320px', display='' if cb_resample.value else 'none')
)

def _toggle_resample(change):
    txt_target_freq.layout.display = '' if change['new'] else 'none'

cb_resample.observe(_toggle_resample, names='value')

# ---- Optional high-pass (analysis only; off by default) ----
# EDFs are usually already AC-coupled (their hardware high-pass is shown in the acquisition
# line above). Applying one here is mainly to HARMONISE a heterogeneous dataset to a common
# corner — choose a target >= the max acquisition high-pass shown. Applied after resampling.
hp_check = widgets.Checkbox(
    value=False, description='Activate high-pass',
    style={'description_width': 'initial'}
)
hp_freq = widgets.BoundedFloatText(
    value=0.5, min=0.01, max=5.0, step=0.05,
    description='High-pass (Hz) :',
    style={'description_width': '180px'},
    # Initial visibility follows the checkbox default (the Curry twin ticks it ON), otherwise
    # the box would stay hidden until the user toggles the checkbox off and on again.
    layout=widgets.Layout(width='320px', display='' if hp_check.value else 'none')
)

def _toggle_highpass(change):
    hp_freq.layout.display = '' if change['new'] else 'none'

hp_check.observe(_toggle_highpass, names='value')


def thresh_row(w, desc):
    return widgets.HBox([
        w,
        widgets.HTML(
            f'<small style="color:#555;margin-left:12px;line-height:32px;">{desc}</small>'
        )
    ])


display(
    HTML('<h3 style="margin-top:4px;">&#128193; Data</h3>'),
    fc_folder,
    hypno_suffix,
    hypno_suffix_info,
    custom_stages_box,
    custom_stages_info,
    fc_config,
    existing_reports_info,
    participant_selector.widget,
    HTML('<h3 style="margin-top:20px;">&#128295; Signal preparation '
         '<small style="color:#888;font-weight:normal;">(applied for this analysis only, not saved)</small></h3>'),
    acq_info,
    widgets.HBox([cb_resample, txt_target_freq]),
    widgets.HBox([hp_check, hp_freq]),
    HTML(
        '<h3 style="margin-top:20px;">&#9881; Quality thresholds</h3>'
        '<small style="color:#555;">Channels exceeding any threshold will be flagged '
        'in the report and in <em>dataset_overview.html</em>.</small>'
    ),
    widgets.VBox([
        thresh_row(thresh_flat,
                   'Fraction of consecutive equal samples: detects flat segments and dead channels'),
        thresh_row(thresh_bounds,
                   'Fraction at EDF physical-range limits: detects hard saturation (declared range)'),
        thresh_row(thresh_peaks,
                   'Peaks in amplitude distribution, ≥2: irregular / suspect distribution (bimodal or multimodal, inspect the channel)'),
        thresh_row(thresh_std_low,
                   'Signal std dev: very low std (< 5 µV) flags a dead or disconnected channel'),
        thresh_row(thresh_hist_extreme,
                   'Fraction in outermost histogram bins: detects in-range clipping (within EDF range)'),
    ])
)

In [ ]:
btn_run = widgets.Button(
    description='▶  Run',
    button_style='primary',
    layout=widgets.Layout(width='200px', height='40px')
)
progress = widgets.IntProgress(
    min=0, max=1, value=0, bar_style='info',
    layout=widgets.Layout(width='500px')
)
progress_label = widgets.Label(value='')

# --- Pipeline progress (second bar): spans ONE participant's whole run, so a long file is
# not mistaken for a crash. Its scale is in arbitrary 'time-cost' units (below) so the bar
# fill AND the 3-segment legend under it are sized ~proportionally to each phase's duration.
# Loading is a one-time cost; the per-channel analysis and (heavier) report-rendering costs
# scale with the channel count. Tune these if the segment widths feel off for your data.
COST_LOAD_DATA    = 8    # EDF read — one-time per file
COST_HIGHPASS     = 3    # high-pass filter — one-time, only when enabled
COST_RESAMPLE     = 4    # resampling — one-time, only when enabled
COST_ANALYSE_CH   = 1    # per channel — metric / histogram computation
COST_SPECTRO_CH   = 3    # per channel — display filter + spectrogram (the slowest step)
COST_RENDER_CH    = 2    # per channel — report figures (spectrogram already computed)


def build_phase_legend(cost_load, cost_analyse, cost_report):
    """3-segment strip drawn under the pipeline bar; segment widths ≈ each phase's share of
    the run time (Load / Per-channel / Report). Rebuilt per participant (widths depend on
    the channel count), and aligned to the 500px bar so the fill matches the segments."""
    total = max(1, cost_load + cost_analyse + cost_report)
    def seg(cost, label, bg, tip, last=False):
        border = '' if last else 'border-right:1px solid #bbb;'
        return (f'<div title="{tip}" style="width:{100 * cost / total:.1f}%;{border}'
                f'text-align:center;background:{bg};overflow:hidden;white-space:nowrap;">{label}</div>')
    return (
        '<div style="display:flex;width:500px;height:14px;font-size:9px;line-height:14px;'
        'color:#444;border:1px solid #bbb;border-top:none;border-radius:0 0 3px 3px;overflow:hidden;">'
        + seg(cost_load, 'Load', '#e7ecff', 'Loading + resampling')
        + seg(cost_analyse, 'Per-channel', '#e7f7e7', 'Per-channel analysis')
        + seg(cost_report, 'Report', '#ffeaea', 'Report building', last=True)
        + '</div>'
    )


progress_ch = widgets.IntProgress(
    min=0, max=1, value=0, bar_style='',
    layout=widgets.Layout(width='500px', margin='0')
)
progress_ch_label = widgets.Label(value='')
# margin='0' + the legend glued directly under the bar (see the display VBox) so the 3 phase
# segments read as the bar's own labelled track.
phase_legend = widgets.HTML(value='', layout=widgets.Layout(margin='0'))
out = widgets.Output()


def _print_out(msg):
    """A message in the run's output area (the quality_lib functions report through it)."""
    with out:
        print(msg)


def _set_status(text):
    progress_ch_label.value = text


def run_analysis(btn):
    btn.disabled = True
    out.clear_output()
    try:

        # --- Validate inputs ---
        if not fc_folder.selected:
            with out:
                print('ERROR: Please select a study folder.')
            return
        if not fc_config.selected:
            with out:
                print('ERROR: Please select the remap/reref config JSON.')
            return

        data_folder = Path(fc_folder.selected)
        config_path = Path(fc_config.selected)
        reports_root = data_folder / 'reports_quality_overview'
        reports_root.mkdir(exist_ok=True)
        custom_stages = parse_custom_field(custom_stages_box.value)

        try:
            with open(config_path, 'r', encoding='utf-8') as f:
                config_dict = json.load(f)
        except Exception as e:
            with out:
                print(f'ERROR loading config: {e}')
            return

        edf_files = [f for f in sorted(data_folder.rglob('*')) if f.suffix.lower() == '.edf' and not f.name.startswith('._')]
        if not edf_files:
            with out:
                print(f'No EDF files found in {data_folder}')
            return

        progress.max = len(edf_files)
        progress.value = 0
        hypno_lookup = {os.path.normcase(p.name): p for p in data_folder.rglob('*') if p.suffix.lower() == '.txt'}
        # Only the participants chosen in the selector (all, or a subset; minus the already-processed
        # ones while 'Skip already processed' is ticked). Excluded participants are processed too.
        if participant_selector.warning_html():
            with out:
                display(HTML(participant_selector.warning_html()))
        _run_ids = {PSL.id_key(i) for i in participant_selector.selected_ids()}
        edf_files = [f for f in edf_files if PSL.id_key(f.stem) in _run_ids]
        progress.max = max(len(edf_files), 1)
        _reg, _reg_warn = PSL.load_registry(data_folder)
        reg_status = PSL.effective_status(_reg)
        if _reg_warn:
            with out:
                print(f'⚠ {_reg_warn}')

        thresholds = {
            'flat_pct': thresh_flat.value,
            'bounds_pct': thresh_bounds.value,
            'n_peaks': thresh_peaks.value,
            'std_low': thresh_std_low.value,
            'hist_extreme_pct': thresh_hist_extreme.value,
        }
        do_resample = cb_resample.value
        target_freq = int(txt_target_freq.value) if do_resample else None
        do_highpass = hp_check.value
        hp_freq_val = float(hp_freq.value) if do_highpass else None
        # Build per-report interpretations: threshold-based metrics show the actual
        # threshold value used for this run (set manually in the notebook widgets).
        interpretations = dict(METRIC_INTERPRETATIONS)
        interpretations.update({
            'Flat signal (%)': f'flag if&nbsp;&gt;&nbsp;{thresholds["flat_pct"]:g}%&nbsp;&mdash; threshold defined manually in the notebook ; typical threshold is ??',
            'At EDF bounds (%)': f'flag if&nbsp;&gt;&nbsp;{thresholds["bounds_pct"]:g}%&nbsp;&mdash; threshold defined manually in the notebook ; typical threshold is ??',
            'Extreme histogram (%)': f'flag if&nbsp;&gt;&nbsp;{thresholds["hist_extreme_pct"]:g}%&nbsp;&mdash; threshold defined manually in the notebook ; typical threshold is ??',
        })

        rows_summary = []
        rows_stage_summary = []
        failed = []
        attempted_ids = set()  # file_ids actually processed (not skipped)
        t_start = time.time()

        for i, edf_path in enumerate(edf_files):
            file_id = edf_path.stem
            progress.value = i
            progress_label.value = f'Participant {file_id}  ({i + 1}/{len(edf_files)})'
            progress_ch.value = 0
            progress_ch_label.value = ''
            phase_legend.value = ''

            # Mirror the EDF subfolder structure under reports_root.
            edf_relative = edf_path.parent.relative_to(data_folder)
            edf_reports_dir = reports_root / edf_relative
            report_path = edf_reports_dir / f'{file_id}_quality_overview.html'
            metrics_path = edf_reports_dir / f'{file_id}_quality_metrics.tsv'

            # Skip only when BOTH the report and the per-file metrics TSV exist, so a run
            # interrupted between the two never leaves a file permanently skipped with its
            # rows missing from quality_summary.tsv. A mismatch is warned about + reprocessed.
            _has_report = report_path.exists()
            _has_data   = metrics_path.exists()
            if skip_existing.value and _has_report and _has_data:
                with out:
                    print(f'⤼ {file_id} ({i + 1}/{len(edf_files)}): skipped (report + data exist)')
                continue
            if skip_existing.value and (_has_report != _has_data):
                with out:
                    print(f'⚠ {file_id} — '
                          + ('report exists but metrics table missing' if _has_report else 'metrics table exists but report missing')
                          + ' — reprocessing to restore consistency.')

            attempted_ids.add(file_id)
            edf_reports_dir.mkdir(parents=True, exist_ok=True)

            # --- Load hypnogram ---
            progress_ch_label.value = f'{file_id} · loading hypnogram…'
            _hypno_name = f'{file_id}{hypno_suffix.value}'
            hypno_path = hypno_lookup.get(os.path.normcase(_hypno_name), edf_path.parent / _hypno_name)
            expert_hypno = None
            hypno_vec = None
            hypno_warning = None
            hypno_warning_plain = None
            if not hypno_path.exists():
                hypno_warning = (
                    f'Hypnogram not found: <b>{hypno_path.name}</b>. '
                    f'Spectrogram skipped. Possible causes: '
                    f'(1) the <i>3_remap_hypno_voila</i> notebook was not run or did not complete; '
                    f'(2) the hypnogram suffix above does not match the actual file suffix.'
                )
                hypno_warning_plain = f'Hypnogram not found: {hypno_path.name} (spectrogram skipped): check that the suffix is correct or that 3_remap_hypno_voila was run.'
            else:
                try:
                    expert_hypno = np.loadtxt(str(hypno_path), dtype=str).astype('<U10')
                    hypno_vec, hypno_warning, hypno_warning_plain = hypno_vector(expert_hypno, custom_stages)
                except Exception as e:
                    hypno_warning = f'Error loading hypnogram: {e}'
                    hypno_warning_plain = f'Error loading hypnogram: {e}'

            # --- Check config ---
            if file_id not in config_dict:
                with out:
                    print(f'WARNING: {file_id} not in config JSON, skipping.')
                failed.append({'file_id': file_id, 'reason': 'not in config JSON'})
                continue

            sub_config = config_dict[file_id]
            # include= utilise les noms d'ORIGINE de l'EDF (les clés du remap), évalué à la
            # LECTURE. On ne garde ainsi que les canaux du montage : la fréquence native de
            # l'EEG est préservée (pas de suréchantillonnage vers un canal hors-montage plus
            # rapide comme un ECG 512 Hz), et on évite un AssertionError de lecture partielle
            # du lecteur EDF de MNE. Même motif que 7_preprocessing_voila.
            selected_channels = list(sub_config['remap'].keys())

            # Set up the pipeline bar for this participant. Channel count is provisional here
            # (from the config); it is corrected to the real count once the EDF is loaded,
            # just before the per-channel analysis. Costs come from the constants above.
            _n_ch_est = len(selected_channels)
            _cost_load = COST_LOAD_DATA + (COST_HIGHPASS if do_highpass else 0) + (COST_RESAMPLE if do_resample else 0)
            progress_ch.max = _cost_load + (COST_ANALYSE_CH + COST_SPECTRO_CH + COST_RENDER_CH) * _n_ch_est
            progress_ch.value = 0
            phase_legend.value = build_phase_legend(_cost_load, (COST_ANALYSE_CH + COST_SPECTRO_CH) * _n_ch_est, COST_RENDER_CH * _n_ch_est)

            progress_ch_label.value = f'{file_id} · loading EDF signal…'
            # --- Load EDF ---
            try:
                raw = mne.io.read_raw_edf(
                    str(edf_path), preload=True, encoding='latin-1',
                    include=selected_channels, verbose=False
                )
            except Exception as e:
                with out:
                    print(f'ERROR loading {file_id}: {e}')
                failed.append({'file_id': file_id, 'reason': f'EDF loading: {e}'})
                continue

            # --- Save physical bounds before channel manipulation ---
            extras = raw._raw_extras[0]
            phys_bounds_by_name = {}
            for idx, ch in enumerate(raw.ch_names):
                base = ch.rsplit('-', 1)[0] if (ch.endswith('-0') or ch.endswith('-1')) else ch
                phys_bounds_by_name[base] = get_phys_bounds_uV(extras, idx)

            # --- Drop suffix duplicates, build remap, rename ---
            raw, _ = drop_suffix_duplicates(raw)
            remap_adapted = adapt_remap_dict_to_suffixes(raw, sub_config['remap'])
            final_to_orig = {}
            for orig, new in remap_adapted.items():
                base = orig.rsplit('-', 1)[0] if (orig.endswith('-0') or orig.endswith('-1')) else orig
                final_to_orig[new] = base
            raw.rename_channels(remap_adapted)
            sf = raw.info['sfreq']
            progress_ch.value = COST_LOAD_DATA   # segment 1: data loaded

            # --- Optional resampling, then high-pass (quality_lib.prepare_signal: tool 6's order) ---
            sf = prepare_signal(raw, sf, file_id, do_resample, target_freq, do_highpass, hp_freq_val,
                                warn=_print_out, status=_set_status)
            progress_ch.value = _cost_load   # segment 1: loading + optional resampling / high-pass done

            # --- Hypnogram length check ---
            expert_hypno, hypno_vec, hypno_warning, hypno_warning_plain = check_hypno_length(
                raw, sf, expert_hypno, hypno_vec, hypno_warning, hypno_warning_plain)

            # Finalize the pipeline bar now the real channel count is known (rebuild the legend
            # if it differs from the config estimate). Analysis fills segment 2, one tick/channel.
            n_ch = len(raw.ch_names)
            # Middle phase = per-channel metrics, then the spectrogram pass (both run before
            # the report is built), so the 3-segment legend still matches the three phases.
            _cost_analyse_metrics = COST_ANALYSE_CH * n_ch
            _cost_analyse = _cost_analyse_metrics + COST_SPECTRO_CH * n_ch
            _cost_render = COST_RENDER_CH * n_ch
            progress_ch.max = _cost_load + _cost_analyse + _cost_render
            phase_legend.value = build_phase_legend(_cost_load, _cost_analyse, _cost_render)
            progress_ch.value = _cost_load

            def _on_metrics_channel(ci, ch):
                progress_ch.value = _cost_load + COST_ANALYSE_CH * (ci + 1)
                progress_ch_label.value = f'{file_id} · analysing {ch} ({ci + 1}/{n_ch})'

            # --- Metrics of every channel (unfiltered signal), whole night + per stage ---
            ch_metrics, ch_flags, file_rows, file_stage_rows = compute_channel_metrics(
                raw, sf, file_id, thresholds, hypno_vec, custom_stages, phys_bounds_by_name, final_to_orig,
                on_channel=_on_metrics_channel)

            # Shared time-series / histogram amplitude limits
            y_lim_ts, x_lim_hist, hist_y_max = display_limits(ch_metrics, ch_flags)

            def _on_spectro_channel(ci, ch, done):
                if done:
                    progress_ch.value = _cost_load + _cost_analyse_metrics + COST_SPECTRO_CH * (ci + 1)
                else:
                    progress_ch_label.value = f'{file_id} · spectrogram {ch} ({ci + 1}/{n_ch})'

            # --- Spectrograms: computed once here, used twice (per channel + median) ---
            ch_sxx, sxx_f, sxx_t = compute_channel_spectrograms(raw, sf, file_id, hypno_vec,
                                                                warn=_print_out, on_channel=_on_spectro_channel)

            progress_ch_label.value = f'{file_id} · building overview…'
            # --- Build mne.Report ---
            report = mne.Report(title=f'{file_id}: Quality Overview', verbose=False)

            # Overview section: channel flag summary
            flagged = {ch: f for ch, f in ch_flags.items() if f}
            if flagged:
                items = ''.join(
                    f'<li><b>{ch}</b>: {", ".join(f)}</li>' for ch, f in flagged.items()
                )
                flag_html = (
                    '<div style="background:#f8d7da;padding:10px;'
                    'border-left:4px solid #dc3545;border-radius:4px;margin-bottom:12px;">'
                    f'<b>⚠ Suspicious channels ({len(flagged)}/{len(raw.ch_names)}):'
                    f'</b><ul>{items}</ul></div>'
                )
            else:
                flag_html = (
                    '<div style="background:#d4edda;padding:10px;'
                    'border-left:4px solid #28a745;border-radius:4px;margin-bottom:12px;">'
                    '<b>✓ All channels within thresholds</b></div>'
                )
            if hypno_warning:
                flag_html += (
                    '<div style="background:#fff3cd;padding:8px;'
                    'border-left:4px solid #ffc107;border-radius:4px;margin-bottom:12px;">'
                    f'⚠ {hypno_warning}</div>'
                )
            _excl, _excl_reason = PSL.status_of(reg_status, file_id)
            if _excl:
                flag_html = PSL.excluded_banner_html(_excl_reason) + flag_html
            report.add_html(html=flag_html, title='Overview', section='Overview')

            # Overview overlays: all channels superimposed (one colour each) + across-channel table.
            try:
                fig_ov_ts = plot_overlay_timeseries(raw, sf, y_lim=y_lim_ts)
                report.add_figure(fig=fig_ov_ts, title='All channels: time series (butterfly)',
                                  section='Overview', image_format='PNG')
                plt.close(fig_ov_ts)
            except Exception as _ov_err:
                with out:
                    print(f'  \u26a0 {file_id}: overview time-series failed: {_ov_err}')
            try:
                fig_ov_psd = plot_overlay_psd(raw, sf)
                report.add_figure(fig=fig_ov_psd, title='All channels: PSD',
                                  section='Overview', image_format='PNG')
                plt.close(fig_ov_psd)
            except Exception as _ov_err:
                with out:
                    print(f'  \u26a0 {file_id}: overview PSD failed: {_ov_err}')

            # Across-channel median spectrogram: one summary view of the night's spectral
            # structure for the whole montage. All channels are kept (flagged ones included,
            # like every other Overview plot); the median is what keeps that readable.
            if ch_sxx:
                try:
                    _n_flagged = sum(1 for _c in ch_sxx if ch_flags.get(_c))
                    fig_ov_spec = plot_median_hypnospectrogram(
                        sxx_f, sxx_t, list(ch_sxx.values()), expert_hypno, custom_stages)
                    fig_ov_spec.set_size_inches(10, 3.5)
                    for ax in fig_ov_spec.axes:
                        ax.tick_params(labelsize=8)
                        ax.xaxis.label.set_fontsize(11)
                        ax.yaxis.label.set_fontsize(11)
                    fig_ov_spec.tight_layout()
                    report.add_figure(
                        fig=fig_ov_spec,
                        title=f'All channels: spectrogram (median of {len(ch_sxx)} channels, '
                              f'{_n_flagged} flagged, all kept)',
                        section='Overview', image_format='PNG')
                    plt.close(fig_ov_spec)
                except Exception as _ov_err:
                    with out:
                        print(f'  \u26a0 {file_id}: overview median spectrogram failed: {_ov_err}')

            try:
                fig_ov_dist = plot_overlay_distribution(raw, x_lim=x_lim_hist)
                report.add_figure(fig=fig_ov_dist, title='All channels: amplitude distribution',
                                  section='Overview', image_format='PNG')
                plt.close(fig_ov_dist)
            except Exception as _ov_err:
                with out:
                    print(f'  \u26a0 {file_id}: overview distribution failed: {_ov_err}')
            try:
                report.add_html(html=build_avg_metrics_html(ch_metrics, raw.ch_names),
                                title='Metrics averaged across channels', section='Overview')
            except Exception as _ov_err:
                with out:
                    print(f'  \u26a0 {file_id}: overview metrics table failed: {_ov_err}')

            # All-electrodes figures (per-participant analogue of dataset_overview): a pooled
            # distribution across this recording channels + a per-electrode scatter, both
            # built from the per-channel rows already collected above.
            try:
                df_sub = pd.DataFrame(file_rows)
                df_stage_sub = pd.DataFrame(file_stage_rows) if file_stage_rows else None
                fig_ov_pool = plot_participant_pooled_boxplots(df_sub, df_stage_sub, custom_stages)
                report.add_figure(
                    fig=fig_ov_pool,
                    title='All electrodes: metric distribution (pooled across channels)',
                    section='Overview', image_format='PNG')
                plt.close(fig_ov_pool)
            except Exception as _ov_err:
                with out:
                    print(f'  \u26a0 {file_id}: overview pooled boxplots failed: {_ov_err}')
            try:
                fig_ov_elec = plot_participant_electrode_scatter(pd.DataFrame(file_rows), custom_stages)
                report.add_figure(fig=fig_ov_elec, title='All electrodes: metric by electrode',
                                  section='Overview', image_format='PNG')
                plt.close(fig_ov_elec)
            except Exception as _ov_err:
                with out:
                    print(f'  \u26a0 {file_id}: overview by-electrode scatter failed: {_ov_err}')

            # One section per channel: histogram + PSD + time series + spectrogram + metrics table.
            # Report = segment 3 of the pipeline bar: it advances once per rendered channel (the
            # bar keeps moving forward continuously — no per-channel reset like the old two-bar UI).
            _report_base = _cost_load + _cost_analyse
            for ci, ch in enumerate(raw.ch_names):
                progress_ch_label.value = f'{file_id} · building report {ch} ({ci + 1}/{n_ch})'
                progress_ch.value = _report_base + COST_RENDER_CH * (ci + 1)
                m = ch_metrics[ch]
                flags = ch_flags[ch]
                sig_uV = raw.get_data(picks=[ci])[0] * 1e6

                # Flagged channels use their own auto-scale; healthy channels share hist_y_max.
                y_max_ch = hist_y_max if not flags else None
                try:
                    fig = plot_histogram_figure(m, ch, y_max=y_max_ch, x_lim=x_lim_hist)
                    report.add_figure(fig=fig, title=f'{ch}: histogram', section=ch,
                                      image_format='PNG')
                    plt.close(fig)
                except Exception as _hist_err:
                    report.add_html(html=f'<p>Histogram generation failed: {_hist_err}</p>',
                                    title=f'{ch}: histogram', section=ch)
                    with out:
                        print(f'  ⚠ {file_id}/{ch}: histogram error: {_hist_err}')

                try:
                    fig_psd = plot_psd_figure(sig_uV, sf)
                    report.add_figure(fig=fig_psd, title=f'{ch}: PSD', section=ch,
                                      image_format='PNG')
                    plt.close(fig_psd)
                except Exception as _psd_err:
                    report.add_html(
                        html=f'<p>PSD generation failed: {_psd_err}</p>',
                        title=f'{ch}: PSD', section=ch
                    )
                    with out:
                        print(f'  ⚠ {file_id}/{ch}: PSD error: {_psd_err}')

                try:
                    fig_ts = plot_timeseries_figure(sig_uV, sf, y_lim_ts)
                    report.add_figure(fig=fig_ts, title=f'{ch}: time series', section=ch,
                                      image_format='PNG')
                    plt.close(fig_ts)
                except Exception as _ts_err:
                    report.add_html(html=f'<p>Time series generation failed: {_ts_err}</p>',
                                    title=f'{ch}: time series', section=ch)
                    with out:
                        print(f'  ⚠ {file_id}/{ch}: time series error: {_ts_err}')

                # Spectrogram: reuses the dB spectrogram computed in the pass above (same array,
                # 0.1–40 Hz bandpass, just computed earlier), so it is never computed twice.
                if hypno_vec is not None and ch in ch_sxx:
                    try:
                        fig_spec = render_hypnospectrogram(
                            sxx_f, sxx_t, ch_sxx[ch].astype(np.float64), expert_hypno, custom_stages)
                        fig_spec.set_size_inches(10, 3.5)
                        for ax in fig_spec.axes:
                            ax.tick_params(labelsize=8)
                            ax.xaxis.label.set_fontsize(11)
                            ax.yaxis.label.set_fontsize(11)
                        fig_spec.tight_layout()
                        report.add_figure(fig=fig_spec, title=f'{ch}: spectrogram', section=ch,
                                          image_format='PNG')
                        plt.close(fig_spec)
                    except Exception as _spec_err:
                        report.add_html(
                            html=f'<p>Spectrogram generation failed: {_spec_err}</p>',
                            title=f'{ch}: spectrogram', section=ch
                        )
                        with out:
                            print(f'  ⚠ {file_id}/{ch}: spectrogram error: {_spec_err}')
                elif hypno_vec is None:
                    report.add_html(
                        html='<p>No spectrogram: hypnogram missing or incompatible with EEG length.</p>',
                        title=f'{ch}: spectrogram', section=ch
                    )
                else:
                    report.add_html(
                        html='<p>No spectrogram: its computation failed for this channel '
                             '(see the notebook output).</p>',
                        title=f'{ch}: spectrogram', section=ch
                    )

                flag_bg = '#f8d7da' if flags else '#f0f0f0'
                flag_text = '; '.join(flags) if flags else 'no flags'
                rows_html = ''.join([
                    f'<tr><td style="padding:4px 10px;">{label}</td>'
                    f'<td style="padding:4px 10px;text-align:right;">{val}</td>'
                    f'<td style="padding:4px 10px;color:#555;font-size:0.85em;">'
                    f'{interpretations.get(label, "")}</td></tr>'
                    for label, val in [
                        ('Mean', f"{m['mean_uV']:.2f} µV"),
                        ('Std dev', f"{m['std_uV']:.2f} µV"),
                        ('p99 |amplitude|', f"{m['p99_abs_uV']:.1f} µV"),
                        ('p99.9 |amplitude|', f"{m['p999_abs_uV']:.1f} µV"),
                        ('Kurtosis (Fischer)', f"{m['kurtosis']:.2f}"),
                        ('Skewness', f"{m['skewness']:.3f}"),
                        ('Flat signal (%)', f"{m['flat_pct']:.2f}%"),
                        ('At EDF bounds (%)', f"{m['bounds_pct']:.3f}%"),
                        ('Extreme histogram (%)', f"{m['hist_extreme_pct']:.3f}%"),
                        ('n_peaks (distribution)', str(m['n_peaks'])),
                    ]
                ])
                metrics_html = (
                    f'<div style="background:{flag_bg};padding:8px;border-radius:4px;'
                    f'margin-bottom:8px;"><b>Flags:</b> {flag_text}</div>'
                    '<table style="border-collapse:collapse;font-size:0.9em;width:100%;">'
                    '<tr style="background:#eee;">'
                    '<th style="padding:4px 10px;text-align:left;">Metric</th>'
                    '<th style="padding:4px 10px;text-align:right;">Value</th>'
                    '<th style="padding:4px 10px;text-align:left;">Interpretation</th></tr>'
                    f'{rows_html}</table>'
                )
                report.add_html(html=metrics_html, title=f'{ch}: metrics', section=ch)

            # --- Write per-file TSVs BEFORE the HTML report ---
            # These per-file tables are the durable cumulative data: quality_summary.tsv is
            # rebuilt from all of them by glob after the loop. Writing them before report.save
            # guarantees 'report exists => data exists', so the skip gate is interruption-safe.
            try:
                if file_rows:
                    pd.DataFrame(file_rows).to_csv(str(metrics_path), sep='\t', index=False)
                if file_stage_rows:
                    pd.DataFrame(file_stage_rows).to_csv(
                        str(edf_reports_dir / f'{file_id}_quality_by_stage.tsv'), sep='\t', index=False)
            except Exception as _e:
                with out:
                    print(f'  ⚠ {file_id}: could not write per-file metrics TSV: {_e}')
            # Keep the in-memory accumulation for the end-of-run summary counts below.
            rows_summary.extend(file_rows)
            rows_stage_summary.extend(file_stage_rows)

            progress_ch_label.value = f'{file_id} · saving report…'
            report.save(str(report_path), overwrite=True, open_browser=False, verbose=False)
            n_flagged = sum(1 for f in ch_flags.values() if f)
            progress_ch_label.value = f'{file_id} · done ✓'
            with out:
                print(f'✓ {file_id}: processed ({n_flagged}/{len(raw.ch_names)} channel(s) flagged)  →  {report_path.name}')
                if hypno_warning_plain:
                    print(f'  ⚠ {hypno_warning_plain}')

        # --- Rebuild cumulative TSVs from the per-file tables on disk ---
        # quality_summary.tsv and quality_summary_by_stage.tsv are regenerated from ALL
        # per-file *_quality_metrics.tsv / *_quality_by_stage.tsv found under reports_root
        # (any subfolder), so a file processed in an earlier run and skipped in this one is
        # never dropped from the cumulative tables. df_new / df_stage_new (this run's rows)
        # are kept below only for the end-of-run summary counts.
        df_new = pd.DataFrame(rows_summary)
        df_stage_new = pd.DataFrame(rows_stage_summary)
        summary_path = reports_root / 'quality_summary.tsv'
        stage_summary_path = reports_root / 'quality_summary_by_stage.tsv'

        metrics_frames = []
        for _p in sorted(reports_root.rglob('*_quality_metrics.tsv')):
            try:
                metrics_frames.append(pd.read_csv(str(_p), sep='\t', dtype={'file_id': str}))
            except Exception as _e:
                with out:
                    print(f'  ⚠ could not read {_p.name}: {_e}')
        if metrics_frames:
            df_summary_all = pd.concat(metrics_frames, ignore_index=True).sort_values(['file_id', 'channel']).reset_index(drop=True)
            df_summary_all = PSL.add_exclusion_columns(df_summary_all, reg_status)   # flagged, never dropped
            df_summary_all.to_csv(str(summary_path), sep='\t', index=False)

        stage_frames = []
        for _p in sorted(reports_root.rglob('*_quality_by_stage.tsv')):
            try:
                stage_frames.append(pd.read_csv(str(_p), sep='\t', dtype={'file_id': str}))
            except Exception as _e:
                with out:
                    print(f'  ⚠ could not read {_p.name}: {_e}')
        if stage_frames:
            df_stage_all = pd.concat(stage_frames, ignore_index=True).sort_values(['file_id', 'channel', 'stage']).reset_index(drop=True)
            df_stage_all = PSL.add_exclusion_columns(df_stage_all, reg_status)
            df_stage_all.to_csv(str(stage_summary_path), sep='\t', index=False)

        # Regenerate dataset_overview.html from the full cumulative quality_summary.tsv
        # so the overview reflects all participants processed to date, not just this run.
        if summary_path.exists():
            try:
                _df_stage = (
                    pd.read_csv(str(stage_summary_path), sep='\t', dtype={'file_id': str})
                    if stage_summary_path.exists() else None
                )
                # a dataset-level statistic: the excluded participants are left out (and named)
                _df_sum = pd.read_csv(str(summary_path), sep='\t', dtype={'file_id': str})
                _excluded = {}
                if PSL.COL_EXCLUDED in _df_sum.columns:
                    _ex = _df_sum[_df_sum[PSL.COL_EXCLUDED].astype(bool)]
                    _excluded = dict(zip(_ex['file_id'], _ex[PSL.COL_REASON]))
                    _df_sum = _df_sum[~_df_sum[PSL.COL_EXCLUDED].astype(bool)]
                if _df_stage is not None and PSL.COL_EXCLUDED in _df_stage.columns:
                    _df_stage = _df_stage[~_df_stage[PSL.COL_EXCLUDED].astype(bool)]
                generate_dataset_overview(
                    _df_sum, reports_root, df_stage=_df_stage, custom_stages=custom_stages, excluded=_excluded
                )
            except Exception as _ov_err:
                with out:
                    print(f'⚠ dataset_overview.html generation failed: {_ov_err}')

        failed_path = reports_root / 'failed_files.tsv'
        if failed:
            df_failed_new = pd.DataFrame(failed)
            if failed_path.exists() and attempted_ids:
                # dtype={'file_id': str}: same reason as quality_summary.tsv above.
                df_failed_existing = pd.read_csv(str(failed_path), sep='\t', dtype={'file_id': str})
                df_failed_existing = df_failed_existing[~df_failed_existing['file_id'].astype(str).map(os.path.normcase).isin({os.path.normcase(s) for s in attempted_ids})]
                df_failed = pd.concat([df_failed_existing, df_failed_new], ignore_index=True)
            else:
                df_failed = df_failed_new
            df_failed.to_csv(str(failed_path), sep='\t', index=False)
        elif failed_path.exists() and attempted_ids:
            # A file that was previously failing and was re-attempted successfully: remove its entry.
            # dtype={'file_id': str}: same reason as quality_summary.tsv above.
            df_failed_existing = pd.read_csv(str(failed_path), sep='\t', dtype={'file_id': str})
            df_failed_cleaned = df_failed_existing[~df_failed_existing['file_id'].astype(str).map(os.path.normcase).isin({os.path.normcase(s) for s in attempted_ids})]
            if df_failed_cleaned.empty:
                failed_path.unlink()
            elif len(df_failed_cleaned) < len(df_failed_existing):
                df_failed_cleaned.to_csv(str(failed_path), sep='\t', index=False)

        progress.value = len(edf_files)
        progress_ch.value = progress_ch.max
        progress_label.value = 'Done.'
        progress_ch_label.value = ''

        elapsed = time.time() - t_start
        mins, secs = divmod(elapsed, 60)
        elapsed_str = f'{int(mins)} min {secs:.0f} s' if mins >= 1 else f'{secs:.1f} s'

        with out:
            if not df_new.empty:
                n_files_done = int(df_new['file_id'].nunique())
                n_suspect_participants = int(df_new.groupby('file_id')['exclude'].any().sum())
                n_suspect = int(df_new['exclude'].sum())
            else:
                n_files_done = n_suspect_participants = n_suspect = 0
            avg_str = f'{elapsed / n_files_done:.0f} s / participant' if n_files_done > 0 else 'N/A'
            print(f'\n=== Analysis complete ===')
            print(f'Output folder              : {reports_root}')
            if (reports_root / 'dataset_overview.html').exists():
                print(f'Dataset overview           : dataset_overview.html')
            print(f'Participants processed     : {n_files_done}')
            print(f'Participants with ≥1 flag  : {n_suspect_participants} / {n_files_done}')
            print(f'Total flagged channels     : {n_suspect}')
            if failed:
                print(f'Files failed to load       : {len(failed)}')
            print(f'Total time                 : {elapsed_str}')
            print(f'Average per participant    : {avg_str}')

    except Exception as _run_err:
        with out:
            print(f'\nUnexpected error, analysis interrupted: {_run_err}')
    finally:
        btn.disabled = False


btn_run.on_click(run_analysis)

display(
    HTML('<h3 style="margin-top:24px;">&#9654; Run Quality Overview</h3>'),
    btn_run,
    widgets.HBox([progress, progress_label]),
    widgets.HBox([widgets.VBox([progress_ch, phase_legend], layout=widgets.Layout(margin='0')),
                  progress_ch_label], layout=widgets.Layout(align_items='center')),
    out
)